# Import Packages

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import requests
import pandas as pd
# import geopandas as gpd
from shapely.geometry import Point, Polygon
import folium
import json
import time
import numpy as np
import h3
from folium.plugins import HeatMap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import time
import json

In [3]:
sys.path.append('src')

In [47]:
from fetch_data import *
from data_io import *
from data_prep import *
from scoring import *
from threshold_clustering import *
from dbscan_clustering import *
from visualization import *

# Inputs

In [31]:
#### FRONTEND INPUTS ####

# need to update it to circular boundaries based on user defined center and radius

#define boundaries
ATLANTA_BBOX  = [33.64, -84.55, 33.89, -84.29]

#need to make it scalable for more features later on
# array(['police_station', 'grocery_store', 'hospital', 'marta_stop', 'school', 'restaurant'], dtype=object)

# 3 is very important, 2 is neutral, 1 is not important, 0 not required
user_weights = {
    'police_station': 2,
    'grocery_store': 3,
    'hospital': 1,
    'marta_stop': 1,
    'school': 0,
    'restaurant': 3
}

#define radius of the search area in km, optional - can also define center
user_radius_km = 12

#boolean for whether the user has a vehicle or not
user_has_vehicle = True

# Query Data

In [ ]:
# all_pois = []
# all_pois = query_restaurant_data(ATLANTA_BBOX, all_pois)
# all_pois = query_park_data(ATLANTA_BBOX, all_pois)
# all_pois = query_hospital_and_clinic_data(ATLANTA_BBOX, all_pois)
# print(f"Total POIs fetched: {len(all_pois)}")

# Query Save & Load

In [7]:
name_of_the_file = "combine_datasets"

In [8]:
# save_pois(all_pois, name_of_the_file)

In [9]:
df_pois = load_pois(name_of_the_file)

Loaded 119976
Summary by type:
type
school            102130
marta_stop          9266
hospital            8013
grocery_store        300
police_station       247
restaurant            20
Name: count, dtype: int64


# Basic EDA

In [10]:
df_pois.isnull().sum(axis=0)

type    0
name    0
lat     0
lon     0
dtype: int64

In [11]:
df_pois.groupby('type').agg({'lat':['min','max'] , 'lon':['min','max'], 'type':['count']})
#looks like the data has higher coverage than just Atlanta and Metro Atlanta

lat                    lon                type
                      min        max         min         max   count
type                                                                
grocery_store   33.293267  34.261048  -84.888640  -83.886021     300
hospital       -14.290242  71.297725 -176.640263  145.724472    8013
marta_stop      33.432372  34.105822  -84.669803  -84.083455    9266
police_station  32.845390  34.557793  -85.287257  -83.596301     247
restaurant      33.761162  33.859863  -84.455464  -84.329030      20
school         -14.348924  71.300337 -176.640331  145.784430  102130

In [18]:
df_pois = keep_pois_within_bbox(df_pois, user_radius_km)
df_pois.head()

Filtered POIs from 5964 to 5964 within bbox


,type,name,lat,lon
0,police_station,DEKALB COUNTY MARSHALS OFFICE,33.774070,-84.297214
1,police_station,GEORGIA BUREAU OF INVESTIGATION,33.692882,-84.272514
2,police_station,FULTON COUNTY MARSHALS OFFICE,33.750652,-84.391145
3,police_station,EAST POINT POLICE DEPARTMENT,33.680791,-84.442195
4,police_station,ATLANTA METROPOLITAN COLLEGE CAMPUS POLICE,33.709847,-84.405454


In [19]:
df_pois.groupby('type').agg({'lat':['min','max'] , 'lon':['min','max'], 'type':['count']})

lat                   lon             type
                      min        max        min        max count
type                                                            
grocery_store   33.683241  33.856565 -84.513404 -84.269429    45
hospital        33.680272  33.856443 -84.511055 -84.274107    23
marta_stop      33.640974  33.857078 -84.517977 -84.258003  5673
police_station  33.654056  33.849226 -84.514504 -84.266770    45
restaurant      33.761162  33.856650 -84.411843 -84.329030    19
school          33.641596  33.856300 -84.517100 -84.258700   159

In [23]:
df_pois['type'].unique()

array(['police_station', 'grocery_store', 'hospital', 'marta_stop',
       'school', 'restaurant'], dtype=object)

# Data Prep - Data Points

In [22]:
# either use boundaries or radius function to create hex grids
# hexagons = create_hex_grids_with_boundaries(df_pois)
hexagons = create_hex_grids_with_radius(df_pois, radius_km=user_radius_km, size_of_grid = 8)



Using circular boundary: center (33.7490, -84.3880), radius 12 km
Generated 890 hexagons (before filtering)
Filtered to 536 hexagons within 12 km of center


# Data Prep - Features

In [26]:
## define config for each POI type
# array(['police_station', 'grocery_store', 'hospital', 'marta_stop', 'school', 'restaurant'], dtype=object)

# define levels for each POI Type
# required to be at a close distance -
# required to be at a moderate distance - 
# required to be at a far distance

poi_types_config = {
    'restaurant': {'types': ['restaurant'], 'decay_rate': 1.5, 'max_distance_km': 10},
    'grocery_store': {'types': ['grocery_store'], 'decay_rate': 2, 'max_distance_km': 8},
    'school': {'types': ['school'], 'decay_rate': 1, 'max_distance_km': 15},
    'hospital': {'types': ['hospital'], 'decay_rate': 0.8, 'max_distance_km': 20},
    'marta_stop': {'types': ['marta_stop'], 'decay_rate': 0.5, 'max_distance_km': 5},
    'police_station': {'types': ['police_station'], 'decay_rate': 0.5, 'max_distance_km': 10},
}

In [ ]:
# will take some time to run
# pass user_has_vehicle to the function later
df_hexagons = calculate_accessibility_scores(hexagons, df_pois)

Calculating accessibility scores for each hexagon...
  Processing hexagon 0/536...
  Processing hexagon 25/536...
  Processing hexagon 50/536...
  Processing hexagon 75/536...
  Processing hexagon 100/536...
  Processing hexagon 125/536...
  Processing hexagon 150/536...
  Processing hexagon 175/536...
  Processing hexagon 200/536...
  Processing hexagon 225/536...
  Processing hexagon 250/536...
  Processing hexagon 275/536...
  Processing hexagon 300/536...
  Processing hexagon 325/536...
  Processing hexagon 350/536...
  Processing hexagon 375/536...
  Processing hexagon 400/536...
  Processing hexagon 425/536...
  Processing hexagon 450/536...
  Processing hexagon 475/536...
  Processing hexagon 500/536...
  Processing hexagon 525/536...

Calculated accessibility scores for 536 hexagons

Accessibility Score Statistics:
       restaurant_accessibility  grocery_store_accessibility  \
count                536.000000                   536.000000   
mean                   0.723259      

In [29]:
#lot's of MARTA stops - sanity check

In [32]:
# df_hexagons = apply_user_weights(df_hexagons, user_weights)


df_hexagons = apply_user_weights(df_hexagons, user_weights, smooth_before_weighting=True, neighbor_weight=0.3)



print(df_hexagons.head())

Applying user preferences: {'police_station': 2, 'grocery_store': 3, 'hospital': 1, 'marta_stop': 1, 'school': 0, 'restaurant': 3}
Normalized weights (exponential): {'police_station': 0.16666666666666666, 'grocery_store': 0.3333333333333333, 'hospital': 0.08333333333333333, 'marta_stop': 0.08333333333333333, 'restaurant': 0.3333333333333333}

Applying spatial smoothing to 5 score columns...
  Smoothing hexagon 0/536...
  Smoothing hexagon 50/536...
  Smoothing hexagon 100/536...
  Smoothing hexagon 150/536...
  Smoothing hexagon 200/536...
  Smoothing hexagon 250/536...
  Smoothing hexagon 300/536...
  Smoothing hexagon 350/536...
  Smoothing hexagon 400/536...
  Smoothing hexagon 450/536...
  Smoothing hexagon 500/536...
Spatial smoothing complete

User Match Score Statistics:
count    536.000000
mean       0.286231
std        0.182803
min        0.018107
25%        0.142938
50%        0.252776
75%        0.388983
max        0.866864
Name: user_match_score, dtype: float64
            

In [33]:
print(df_hexagons['user_match_score'].describe())

count    536.000000
mean       0.286231
std        0.182803
min        0.018107
25%        0.142938
50%        0.252776
75%        0.388983
max        0.866864
Name: user_match_score, dtype: float64


In [34]:
# Top 5 hexagons
top_5 = df_hexagons.nlargest(5, 'user_match_score')
print(top_5[['hex_id', 'user_match_score', 'restaurant_accessibility', 'grocery_store_accessibility']])

# Bottom 5 hexagons
bottom_5 = df_hexagons.nsmallest(5, 'user_match_score')
print(bottom_5[['hex_id', 'user_match_score', 'restaurant_accessibility', 'grocery_store_accessibility']])

              hex_id  user_match_score  restaurant_accessibility  \
447  8844c1a8e1fffff          0.866864                  3.471225   
193  8844c1a8e3fffff          0.850145                  3.528258   
405  8844c1a8e7fffff          0.832709                  3.278217   
40   8844c1a8a9fffff          0.827531                  3.465059   
185  8844c1a9dbfffff          0.821967                  2.631071   

     grocery_store_accessibility  
447                     1.836989  
193                     1.816818  
405                     1.883489  
40                      1.844277  
185                     2.327211  
              hex_id  user_match_score  restaurant_accessibility  \
42   8844c1ab49fffff          0.018107                       0.0   
163  8844c1ab61fffff          0.018327                       0.0   
299  8844c1ab45fffff          0.018393                       0.0   
227  8844c1b9bdfffff          0.019315                       0.0   
280  8844c1b9b5fffff          0.021435   

# Experiment 1: Threshold Clustering

In [36]:
# df_threshold = cluster_based_on_score(df_hexagons)
# df_threshold.head()


df_classified = cluster_based_on_score(
    df_hexagons,
    n_tiers=10
)

Classifying into 10 tiers:
  Tier 0 threshold (top 90.0%): 0.543
  Tier 1 threshold (top 80.0%): 0.433
  Tier 2 threshold (top 70.0%): 0.361
  Tier 3 threshold (top 60.0%): 0.306
  Tier 4 threshold (top 50.0%): 0.253
  Tier 5 threshold (top 40.0%): 0.208
  Tier 6 threshold (top 30.0%): 0.167
  Tier 7 threshold (top 20.0%): 0.120
  Tier 8 threshold (top 10.0%): 0.079

Suitability Distribution:
suitability_label
Less Suitable    53
Most Suitable    54
Okay             54
Name: count, dtype: int64

SUITABILITY TIER CHARACTERISTICS

Most Suitable (54 hexagons):
  Match Score Range: 0.545 - 0.867
  Avg restaurant: 2.314
  Avg grocery_store: 1.551
  Avg school: 15.187

Okay (54 hexagons):
  Match Score Range: 0.433 - 0.541
  Avg restaurant: 1.491
  Avg grocery_store: 1.067
  Avg school: 14.231

Less Suitable (53 hexagons):
  Match Score Range: 0.361 - 0.433
  Avg restaurant: 1.136
  Avg grocery_store: 0.851
  Avg school: 13.933


In [38]:
threshold_map_name = "data/output_data/atlanta_threshold_map.html"
map_threshold = create_suitability_map(df_classified, user_weights)
map_threshold.save(threshold_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...


In [39]:
save_csv(df_classified, "combined_data_outputs")

Saved CSV: data/output_data/combined_data_outputs.csv


In [44]:
df_classified_json = convert_json(df_classified)

In [48]:
save_json(df_classified_json, "combined_data_outputs_json")

# Experiment 2: DBSCAN Clustering

In [14]:
df_dbscan = dbscan_score_clustering(df_hexagons, eps=0.1, min_samples=3)


DBSCAN CLUSTERING (Score-Based)
Parameters: eps=0.1, min_samples=3
Score range after scaling: [0.000, 1.000]

Results:
  Clusters found: 1
  Noise points: 0
  Cluster 0: 536 hexagons, avg score = 0.161


In [15]:
cluster_colors = get_cluster_colors(df_dbscan)

In [16]:
dbscan_map_name = "data/output_data/atlanta_dbscan_map.html"
map_dbscan = create_dbscan_map(df_dbscan, user_weights, cluster_colors=cluster_colors, use_heatmap=True, heatmap_radius=15)
map_dbscan.save(dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...


# Experiment 3 - Sptial DBSCAN

In [17]:
df_dbscan_spatial = dbscan_spatial_clustering(
    df_hexagons,
    eps=0.2,
    min_samples=3,
    spatial_weight=0.4
)


DBSCAN CLUSTERING (Spatially-Aware)
Parameters: eps=0.2, min_samples=3, spatial_weight=0.4

Results:
  Clusters found: 6
  Noise points: 1
  Noise/Uncertain: 1 hexagons
  Region 0: 502 hexagons, avg score = 0.127, extent = 33.1 km
  Region 1: 5 hexagons, avg score = 0.783, extent = 3.7 km
  Region 2: 15 hexagons, avg score = 0.590, extent = 7.3 km
  Region 3: 5 hexagons, avg score = 0.699, extent = 4.9 km
  Region 4: 5 hexagons, avg score = 0.899, extent = 2.6 km
  Region 5: 3 hexagons, avg score = 0.521, extent = 3.4 km


In [18]:
cluster_colors_spatial = get_cluster_colors(df_dbscan_spatial)

In [19]:
map_dbscan_spatial = create_dbscan_map(
    df_dbscan_spatial,
    user_weights,
    cluster_colors=cluster_colors_spatial,
    use_heatmap=True,
    heatmap_radius=15
)

spatial_dbscan_map_name = "data/output_data/atlanta_spatial_dbscan_map.html"
map_dbscan_spatial.save(spatial_dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...
